In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================


# ITDA 3rd 학술제 - 소비기한 OCR 추론 노트북
팀: `DScover_카피바라` (itda3-dscover-capybara)

아키텍처 요약은 팀 저장소의 `[DScover_카피바라]_아키텍처구조도.pdf` 및 `README.md` 를 참고하세요.


## 1. 환경 설정
- CPU 전용, 오프라인 실행을 전제로 합니다 (`gpu=False`, `download_enabled=False`).
- 가중치는 `./weights` 폴더에 사전 배치되어 있어야 합니다 (`download_weights.sh` 참고).
- 4-Core vCPU 채점 환경을 가정하여 `ThreadPoolExecutor` 로 이미지 단위 병렬처리를 수행합니다.
  (Jupyter 커널 내부에서 `multiprocessing.Pool(fork)` 를 쓰면, 커널이 이미 여러 백그라운드
  스레드를 띄운 상태이므로 fork 직후 자식 프로세스가 락을 획득하지 못해 멈추는 경우가 있어
  스레드 기반 병렬화로 대체했습니다.)


In [ ]:
import os, sys, time, re, glob, warnings, threading
from datetime import date as _date
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")

# 각 스레드에서 torch/BLAS가 내부적으로 다시 멀티스레드를 켜서 서로 경쟁(oversubscription)하는
# 것을 막기 위해, 프로세스 전체의 BLAS 스레드 수를 1로 고정한다. (워커 스레드 수만큼만 코어 활용)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

# 일부 numpy/scipy 조합에서는 numpy.testing 모듈이 자체적으로 _blas_supports_fpe 심볼을
# 참조하지만 실제 컴파일된 확장에는 없어 scipy.ndimage import 시점에 크래시가 나는 경우가
# 있다(OCR 기능과 무관한 numpy 내부 테스트 유틸리티 심볼). easyocr이 내부적으로 scipy.ndimage
# 를 import 하므로, 그보다 먼저 더미 값을 채워 넣어 이 문제를 우회한다.
import numpy as _np
try:
    _core_mod = getattr(_np, "_core", None) or getattr(_np, "core", None)
    _umath = getattr(_core_mod, "_multiarray_umath", None) if _core_mod is not None else None
    if _umath is not None and not hasattr(_umath, "_blas_supports_fpe"):
        _umath._blas_supports_fpe = lambda *a, **k: False
except Exception:
    pass

WEIGHTS_DIR = os.path.abspath("./weights")
N_WORKERS = min(4, max(1, os.cpu_count() or 1))   # 채점 환경: Standard 4-Core vCPU

# 명시적 안전 마진 정책: nbconvert 타임아웃(2400s)에서 SAFETY_MARGIN_SEC 만큼 반드시 남기고,
# 그 안에서만 "시간을 더 써서 정확도를 높이는" 최적화를 허용한다. 실제 4-core 환경(Codespaces)
# 실측 결과 모델 로딩·CSV 저장 등 부가 비용은 10초 내외로 작았지만, 채점 서버가 그보다 느릴
# 가능성과 예측 오차에 대비해 250초의 여유를 유지한다. 이 값을 더 줄이는 것은 이번 최적화
# 범위에서 제외한다 (정확도보다 "타임아웃으로 정량 0점"을 피하는 것이 항상 우선).
SAFETY_MARGIN_SEC = 250
TIME_BUDGET_SEC = 2400 - SAFETY_MARGIN_SEC   # 2150

# ── 1·2단계: classical CV 기반 저비용 후보 검출 + 배치 인식 (실제 4-core 환경 실측 근거) ──
# 로컬(2-core) 실측에서는 classical CV 1단계가 2단계 fallback과 비용이 비슷해 폐기했었으나,
# 실제 4-core Codespaces에서 전체 3,352장을 다시 측정해보니 정반대였다: recognize()만 호출하는
# 1단계(후보 16개, 배치 인식)+2단계(실패 시 상위 6개 후보만 대비 보정 재인식) 조합으로 전체를
# 1,475초 만에 다 처리하고 예산(2,250초 기준)의 34%가 남았다 — 이전 우리 방식(9.37%, 54%
# 커버리지)보다 정확도(11.28%)·커버리지(100%) 모두 우수했다. 즉 CRAFT 전체 재검출을 생략하고
# 후보 영역만 인식하는 것 자체가 실제 4-core 환경에서는 훨씬 저렴하다는 뜻이다.
FAST_WORK_WIDTH = 1200
FAST_MAX_BOXES = 16
ROI_RETRY_BOXES = 6

# ── 3단계: 1·2단계가 모두 실패한 이미지에 한해, 남는 시간 예산으로 진짜 전체 재검출 ──
# 1·2단계는 classical CV가 처음부터 후보 박스를 놓치면(예: 각인 위치가 예상 밖) 구조적으로
# 못 찾는다. 이 경우에만, 그리고 시간이 허락할 때만, CLAHE 대비보정 + EasyOCR 전체 파이프라인
# (검출+인식)을 별도 canvas로 1회 더 시도한다. "워커 1개가 이미지 1장에 쓸 수 있는 여유시간"
# = (남은 시간 * N_WORKERS) / 남은 이미지 수 가 클수록 큰 canvas를, 작을수록 작은 canvas를
# 쓰고, 그마저도 안 되면 3단계를 건너뛴다 — 즉 "남는 시간만큼만" 정확도에 재투자하고, 안전
# 마진(SAFETY_MARGIN_SEC) 밑으로는 절대 내려가지 않는다. 임계값은 로컬 2-core 실측(1·2단계와
# 무관하게 별도로 측정한 EasyOCR 전체 파이프라인 처리시간)을 기반으로 하며, 실제 4-core
# 환경에서 재검증을 권장한다.
TIER3_CANVAS_SCHEDULE = [
    (3.6, 550),
    (2.6, 450),
    (1.5, 350),
    (0.8, 250),
]
TIER3_MIN_VIABLE_BUDGET = 0.35   # 이보다도 여유가 없으면 3단계를 생략하고 NONE 유지

print(f"N_WORKERS={N_WORKERS}  WEIGHTS_DIR={WEIGHTS_DIR}")


## 2. 설계 배경 (요약)

상품 뒷면 사진에는 소비기한 외에도 제조일자, 바코드, 품목보고번호, 전화번호, 영양성분,
LOT 번호 등 다양한 숫자가 섞여 있다. 또한 CPU 전용/오프라인/2400초 타임아웃이라는 강한
제약이 있어, 단순히 고해상도로 전체 이미지를 딥러닝 OCR에 넣는 방식은 시간 초과 위험이 크다
(자세한 실험 근거는 아키텍처 요약서 PDF 참고).

이 노트북은 **3단계 구조**를 쓴다 — 앞 두 단계는 값싸게 최대한 많이 맞히고, 3단계는 남는
시간 예산만큼만 정확도에 재투자한다.

1. **1단계 (classical CV 후보 검출 + 배치 인식)**: 형태학적 연산으로 텍스트 줄 후보를 최대
   `FAST_MAX_BOXES`(16)개 찾고, EasyOCR 검출기(CRAFT)는 생략한 채 인식기만 원본 해상도로
   한 번에 배치 호출한다. 처음에는 이 단계를 로컬(2-core) 프로파일링만으로 폐기했었으나(적중률
   8~10%, 처리시간은 2단계와 비슷하다고 측정됨), 실제 4-core Codespaces에서 전체 3,352장을
   다시 측정하니 정반대였다: 1·2단계만으로 전체를 1,475초에 처리하고 정확도(11.28%)도
   당시 우리 방식(9.37%)보다 높았다. 로컬 2-core와 실제 4-core에서 상대적 비용 구조가 다르다는
   뜻이라, 로컬 프로파일링 결과를 실제 채점 환경 검증 없이 최종 결론으로 삼지 않기로 했다.
2. **2단계 (ROI 재시도)**: 1단계가 실패하면, 이미지 전체를 다시 검출하지 않고 1단계가 찾은
   후보 중 상위 `ROI_RETRY_BOXES`(6)개만 CLAHE로 대비를 보정해 재인식한다.
3. **3단계 (전체 재검출, 시간 예산이 허락할 때만)**: 1·2단계 모두 실패하는 경우는 classical CV가
   애초에 후보 박스 자체를 놓친 경우(예: 각인 위치가 예상 밖)라 2단계로는 구조적으로 복구할 수
   없다. 이때 **명시적 안전 마진(`SAFETY_MARGIN_SEC`=250초)을 침범하지 않는 한도 내에서만**
   CLAHE + EasyOCR 전체 파이프라인(검출+인식)을 다시 시도한다. 여유시간(`(남은 시간×N_WORKERS)
   /남은 이미지 수`)이 클수록 큰 canvas를, 작을수록 작은 canvas를 쓰고(`TIER3_CANVAS_SCHEDULE`),
   그마저 안 되면 3단계를 생략해 NONE으로 남긴다. 즉 "시간이 남으면 남을수록 더 정확해지고,
   빠듯해지면 안전 마진을 지키는 선에서 멈추는" 구조다.
4. **키워드 기반 후보 스코어링**: 정규식으로 날짜 형식 후보를 모두 찾은 뒤, 인접 텍스트에
   `소비기한/유통기한/까지` 가 있으면 가점, `제조일자/제조/LOT` 가 있으면 감점하며, 달력 기준
   유효성 검사(`datetime.date`)로 `2026-02-30`처럼 실존하지 않는 날짜는 후보에서 제외한다.
5. **시간 예산 가드레일**: 어느 단계에서든 안전 마진 아래로 내려갈 것으로 판단되면 그 시점
   이후 작업은 즉시 `NONE` 처리하여, 어떤 경우에도 채점 타임아웃(정량 0점)이 발생하지 않도록
   한다.


## 3. 날짜 후보 파서 (정규식 + 키워드 스코어링)

In [ ]:
POS_KEYWORDS = ["소비기한", "유통기한", "품질유지기한", "소비 기한", "유통 기한"]
SUFFIX_KEYWORDS = ["까지", "EXP", "exp", "Best before", "BEST BEFORE", "best before"]
NEG_KEYWORDS = [
    "제조일자", "제조일", "제조년월일", "생산일자", "MFG", "mfg", "제조",
    "PROD.", "PROD DATE", "PRODUCTION DATE", "MFD",
]
LOT_KEYWORDS = ["LOT", "Lot", "lot", "로트"]

DATE_RE = re.compile(
    r"(?<!\d)(20\d{2})\s*[.\-/,년]\s*(\d{1,2})\s*[.\-/,월]\s*(\d{1,2})\s*일?(?!\d)"
)
DATE_RE_COMPACT = re.compile(r"(?<!\d)(20\d{2})(\d{2})(\d{2})(?!\d)")

# ── 수입/해외 제품 대응: 연도가 맨 앞이 아닌 표기 ──────────────────────────
# 국내 제품은 거의 전부 연-월-일 순이라 위 두 패턴으로 커버되지만, 국내에 유통되는
# 해외 제품은 일-월-년(유럽식) 또는 월-일-년(미국식) 표기가 섞여 있다. "이 표기가
# 어느 나라 관습인지"를 추정하는 대신, 달이 12개월뿐이라는 사실만으로 순서를 역산한다
# (_resolve_month_day 참고) — 국가 판별 없이 두 관습을 동시에 커버할 수 있고, 국가
# 추정이 틀릴 위험 자체가 없다.
MONTH_NAMES = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4, "MAY": 5, "JUN": 6,
    "JUL": 7, "AUG": 8, "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12,
}
_MONTH_NAME_PAT = "|".join(MONTH_NAMES.keys())

# 숫자-숫자-연도 (구분자: . - / , 공백). 어느 쪽이 월/일인지는 아직 모른다.
DATE_RE_YEAR_LAST = re.compile(
    r"(?<!\d)(\d{1,2})\s*[.\-/,\s]\s*(\d{1,2})\s*[.\-/,\s]\s*(20\d{2})(?!\d)"
)
# 월 이름이 앞: DEC-28-2021 / DEC 28 2021
DATE_RE_MONTHNAME_FIRST = re.compile(
    r"(?<![A-Za-z])(" + _MONTH_NAME_PAT + r")\s*[-.,\s]\s*(\d{1,2})\s*[-.,\s]\s*(20\d{2})(?![A-Za-z\d])",
    re.IGNORECASE,
)
# 월 이름이 중간: 28-DEC-2021 / 28 DEC 2021
DATE_RE_MONTHNAME_MID = re.compile(
    r"(?<!\d)(\d{1,2})\s*[-.,\s]\s*(" + _MONTH_NAME_PAT + r")\s*[-.,\s]\s*(20\d{2})(?!\d)",
    re.IGNORECASE,
)

# 2자리 연도 표기 (예: 22.10.21 / 20.12.14). 실제 실패 이미지 표본 84장을 조사한 결과
# 3장에서 OCR이 날짜를 정확히 읽고도 "연도가 4자리가 아니라서" 버려지고 있었다.
# 다만 2자리 숫자 3개는 무게·가격·로트번호 등과 형태가 겹쳐 오탐 위험이 크므로,
#   (1) 구분자가 . - / 중 하나로 확실히 있을 것 (공백/콤마 구분은 제외),
#   (2) 어느 자리를 연도로 읽든 2020~2035 범위일 것,
#   (3) 달력상 실존하는 날짜일 것
# 세 조건을 모두 만족할 때만 후보로 인정하고, 4자리 연도 패턴이 하나도 없을 때만 쓴다.
DATE_RE_2DIGIT = re.compile(
    r"(?<!\d)(\d{2})\s*[.\-/]\s*(\d{1,2})\s*[.\-/]\s*(\d{1,2})(?!\d)"
)

# 후보의 "근거 강도" 등급 (숫자가 작을수록 강함). 키워드 점수가 같아 우열을 가릴 수
# 없을 때만 동점 처리 기준으로 쓰인다 — 키워드(소비기한/제조일자) 신호가 항상 우선이다.
RANK_STRONG = 0    # 4자리 연도 + 연/월/일 모두 확정
RANK_2DIGIT = 1    # 2자리 연도 표기를 해석한 것
RANK_PARTIAL = 2   # 일부 필드가 NONE (표기 순서 모호 / 필드 일부 깨짐)

# 날짜 바로 뒤에 "시:분"이 붙어 있으면 생산 타임스탬프(제조 시각)로 본다.
# 실제 실패 이미지 001540 에서 `20.12.14 22:54` 가 관측됐는데, 소비기한은 보통 시각까지
# 찍지 않으므로 이런 형태는 제조일자로 판단해 2자리 연도 후보에서 제외한다. 키워드
# (제조일자/MFG)가 OCR에 안 잡힌 경우에도 동작하는 보조 안전장치다. 콜론이 콤마로
# 오인식되는 경우가 잦아(실측: 22:54 가 `22,54`, `22*54` 로 읽힘) 이런 변형도 시각
# 구분자로 함께 인정한다. 마침표(.)는 진짜 날짜 뒤에 다른 숫자가 오는 경우와 헷갈릴 수
# 있어 제외한다.
# (주의: pattern.match(text, pos) 에서 ^ 앵커는 pos 가 아니라 문자열 맨 앞에만 걸리므로
#  ^ 를 쓰지 않는다. match() 자체가 pos 위치에 고정 매칭한다.)
_TIME_AFTER_RE = re.compile(r"\s*\d{1,2}\s*[:,;*]\s*\d{2}(?!\d)")


def _valid_ymd(y, m, d):
    y, m, d = int(y), int(m), int(d)
    if not (2020 <= y <= 2035):
        return False
    if not (1 <= m <= 12):
        return False
    try:
        _date(y, m, d)   # 2월 30일처럼 달력에 없는 날짜(OCR 오인식)를 제거
    except ValueError:
        return False
    return True


def _valid_year_only(y):
    return 2020 <= int(y) <= 2035


def _resolve_month_day(a, b):
    """두 숫자(a, b) 중 어느 것이 월이고 어느 것이 일인지, 국가 표기 관습을 추정하지
    않고 산술적으로만 판단한다. 달은 12개월뿐이므로 하나가 13~31 범위(=월이 될 수
    없음)면 그게 일(day)로 확정되고 나머지가 월이 된다. 즉 일-월-년(유럽식)이든
    월-일-년(미국식)이든 국가를 몰라도 동일하게 처리된다.
    반환: (month, day) 확정 / "AMBIGUOUS" (둘 다 1~12라 순서를 알 수 없음) /
    None (둘 다 13 이상이라 애초에 유효한 날짜가 아님 → 오탐으로 버림)."""
    a, b = int(a), int(b)
    a_month_ok = 1 <= a <= 12
    b_month_ok = 1 <= b <= 12
    if not (1 <= a <= 31) or not (1 <= b <= 31):
        return None
    if a == b:
        return (a, b)   # 어느 쪽으로 읽어도 같은 날짜이므로 애매하지 않다
    if a_month_ok and not b_month_ok:
        return (a, b)
    if b_month_ok and not a_month_ok:
        return (b, a)
    if a_month_ok and b_month_ok:
        return "AMBIGUOUS"
    return None


# ── 바코드(EAN-13) 국가 힌트: 위에서도 여전히 AMBIGUOUS인 최후의 경우에만 사용 ──────
# 국가/언어를 직접 인식하는 대신, 제품에 인쇄된 EAN-13 바코드의 GS1 국가 접두어로
# "그 나라가 관용적으로 쓰는 날짜 표기 순서"만 참고한다. 체크섬 검증을 통과한
# 13자리 숫자열만 바코드로 인정하므로 전화번호·신고번호 등 임의의 13자리 숫자를
# 오인할 위험은 낮다. 이 힌트는 _resolve_month_day가 "AMBIGUOUS"를 반환한 경우에만
# 쓰이며, 이미 확정 가능한 날짜(>12 규칙, 월 이름, 연-월-일 패턴)는 절대 덮어쓰지
# 않는다.
def _ean13_checksum_ok(code13):
    if len(code13) != 13 or not code13.isdigit():
        return False
    digits = [int(c) for c in code13]
    body, check = digits[:-1], digits[-1]
    total = sum(d if i % 2 == 0 else d * 3 for i, d in enumerate(body))
    return (10 - (total % 10)) % 10 == check


def _barcode_country_bucket(code13):
    """체크섬이 유효한 EAN-13의 GS1 국가 접두어(앞 3자리)로 날짜 표기 순서를 추정한다.
    반환값은 "mdy" / "dmy" / None(불명 또는 이미 연-월-일 관례라 참고할 필요가 없음)뿐이다."""
    if not _ean13_checksum_ok(code13):
        return None
    prefix = int(code13[:3])
    if 0 <= prefix <= 139 or 754 <= prefix <= 755:
        return "mdy"   # 미국/캐나다
    if prefix == 880 or 450 <= prefix <= 459 or 490 <= prefix <= 499 or 690 <= prefix <= 699:
        return None    # 한국/일본/중국: 연-월-일 관례라 DATE_RE 단계에서 이미 처리됨
    if 200 <= prefix <= 899:
        return "dmy"   # 유럽/중남미/중동/아프리카 등 다수
    return None


_BARCODE_DIGIT_RUN_RE = re.compile(r"\d{8,}")


def _detect_country_bucket(ocr_results):
    """이미지의 OCR 조각 전체에서 체크섬이 유효한 EAN-13 숫자열을 찾아 국가 힌트를
    구한다. 8자리 이상 숫자열 안의 모든 13자리 슬라이딩 윈도우를 검사하므로, 바코드
    숫자 앞뒤에 공백 없이 다른 문자가 OCR로 붙어 나와도 탐지할 수 있다."""
    for (box, text, conf) in ocr_results:
        for run in _BARCODE_DIGIT_RUN_RE.findall(text):
            for start in range(0, len(run) - 12):
                bucket = _barcode_country_bucket(run[start:start + 13])
                if bucket is not None:
                    return bucket
    return None


def _resolve_2digit(a, b, c, country_bucket=None):
    """2자리 연도 날짜(예: 22.10.21)를 해석한다. YY.MM.DD / DD.MM.YY / MM.DD.YY 세 가지
    읽기를 모두 시도해 "연도가 2020~2035이고 달력상 실존하는" 해석만 남긴다.
      - 유효한 해석이 하나뿐이면 그것으로 확정한다 (예: 20.12.14 는 맨 뒤 14를 연도로
        읽으면 2014년이 되어 범위를 벗어나므로 YY.MM.DD 하나만 남는다).
      - 둘 이상이면(예: 22.10.21 은 2022-10-21 로도, 2021-10-22 로도 읽힌다) 바코드
        국가 힌트로 고르고, 힌트가 없으면 국내 관례인 YY.MM.DD 를 택한다 — 데이터셋이
        국내 유통 상품 사진이고 한국은 연-월-일 표기가 표준이기 때문이다.
    반환: (year, month, day) 문자열 튜플 또는 None(어떤 해석으로도 유효하지 않음)."""
    a, b, c = int(a), int(b), int(c)
    cands = {}
    if 20 <= a <= 35 and _valid_ymd(2000 + a, b, c):
        cands["ymd"] = (2000 + a, b, c)
    if 20 <= c <= 35 and _valid_ymd(2000 + c, b, a):
        cands["dmy"] = (2000 + c, b, a)
    if 20 <= c <= 35 and _valid_ymd(2000 + c, a, b):
        cands["mdy"] = (2000 + c, a, b)
    if not cands:
        return None
    if len(cands) == 1:
        y, mo, d = next(iter(cands.values()))
    else:
        key = country_bucket if country_bucket in cands else "ymd"
        if key not in cands:
            key = sorted(cands.keys())[0]
        y, mo, d = cands[key]
    return (str(y), str(mo).zfill(2), str(d).zfill(2))


def _salvage_prefix(y, mo, d):
    """연-월-일 형식은 분명한데 일부 필드만 OCR 오인식으로 달력 검증에 실패한 경우
    (예: 2025.05.0 → 일이 깨짐, 2025.71.10 → 월이 깨짐), 전부 버리지 않고 왼쪽부터
    검증되는 필드까지만 살린다: 2025-05-NONE / 2025-NONE-NONE.
    채점이 연/월/일 필드별 부분점수이므로 전부 NONE 으로 버리는 것보다 유리하고,
    빠진 값을 추측하는 게 아니라 "검증된 필드만" 내보내는 것이라 정밀도 우선 원칙과도
    어긋나지 않는다. 깨진 필드의 오른쪽은 자릿수 밀림 가능성이 있어 그 자체로는
    유효해 보여도 신뢰하지 않는다."""
    if not _valid_year_only(y):
        return None
    mo_i, d_i = int(mo), int(d)
    if not (1 <= mo_i <= 12):
        return (y, "NONE", "NONE")
    try:
        _date(int(y), mo_i, d_i)
    except ValueError:
        return (y, str(mo_i).zfill(2), "NONE")
    return None


def find_date_candidates(text, country_bucket=None):
    """연-월-일(국내 표기)을 최우선으로 찾고, 그게 하나도 없을 때만 해외 제품 표기
    (일-월-년 / 월-일-년, 숫자형 또는 영문 월 이름)를 보조로 찾는다. 순서를 판별할
    수 없이 애매한 경우(_resolve_month_day == "AMBIGUOUS")는 country_bucket(바코드
    국가 힌트, extract_best_date에서 이미지 전체 기준으로 한 번만 계산)을 참고하고,
    그래도 판단할 근거가 없으면 연도만 살려 (year, "NONE", "NONE") 형태로 반환한다
    — 정밀도 우선 원칙과 동일.
    """
    out = []
    partial = []   # 일부 필드가 NONE 인 후보 (같은 조각 안에서 완전한 날짜를 못 찾았을 때만)
    for m in DATE_RE.finditer(text):
        y, mo, d = m.group(1), m.group(2), m.group(3)
        if _valid_ymd(y, mo, d):
            out.append((y, mo.zfill(2), d.zfill(2), RANK_STRONG))
        else:
            sv = _salvage_prefix(y, mo, d)
            if sv is not None:
                partial.append((sv[0], sv[1], sv[2], RANK_PARTIAL))
    if out:
        return out
    for m in DATE_RE_COMPACT.finditer(text):
        y, mo, d = m.group(1), m.group(2), m.group(3)
        if _valid_ymd(y, mo, d):
            out.append((y, mo.zfill(2), d.zfill(2), RANK_STRONG))
    if out:
        return out

    for m in DATE_RE_MONTHNAME_FIRST.finditer(text):
        mon_name, d, y = m.group(1), m.group(2), m.group(3)
        mo = MONTH_NAMES.get(mon_name.upper())
        if mo is not None and _valid_ymd(y, mo, d):
            out.append((y, str(mo).zfill(2), d.zfill(2), RANK_STRONG))
    for m in DATE_RE_MONTHNAME_MID.finditer(text):
        d, mon_name, y = m.group(1), m.group(2), m.group(3)
        mo = MONTH_NAMES.get(mon_name.upper())
        if mo is not None and _valid_ymd(y, mo, d):
            out.append((y, str(mo).zfill(2), d.zfill(2), RANK_STRONG))
    if out:
        return out

    for m in DATE_RE_YEAR_LAST.finditer(text):
        a, b, y = m.group(1), m.group(2), m.group(3)
        if not _valid_year_only(y):
            continue
        resolved = _resolve_month_day(a, b)
        if resolved is None:
            continue
        if resolved == "AMBIGUOUS":
            a_i, b_i = int(a), int(b)
            if country_bucket == "mdy" and _valid_ymd(y, a_i, b_i):
                out.append((y, str(a_i).zfill(2), str(b_i).zfill(2), RANK_STRONG))
            elif country_bucket == "dmy" and _valid_ymd(y, b_i, a_i):
                out.append((y, str(b_i).zfill(2), str(a_i).zfill(2), RANK_STRONG))
            else:
                partial.append((y, "NONE", "NONE", RANK_PARTIAL))
        else:
            mo, d = resolved
            if _valid_ymd(y, mo, d):
                out.append((y, str(mo).zfill(2), str(d).zfill(2), RANK_STRONG))
    if out:
        return out

    # 4자리 연도 패턴이 이 조각에 하나도 없을 때만 2자리 연도 표기를 본다.
    for m in DATE_RE_2DIGIT.finditer(text):
        if _TIME_AFTER_RE.match(text, m.end()):
            continue   # `20.12.14 22:54` 같은 생산 타임스탬프는 소비기한이 아니다
        r = _resolve_2digit(m.group(1), m.group(2), m.group(3), country_bucket)
        if r is not None:
            out.append((r[0], r[1], r[2], RANK_2DIGIT))
    if out:
        return out

    # 완전한 날짜를 끝내 못 찾았을 때만, 깨진 필드를 NONE 으로 둔 부분 후보를 쓴다.
    return partial


def _cy(box):
    ys = [p[1] for p in box]
    return sum(ys) / len(ys)


def _cx(box):
    xs = [p[0] for p in box]
    return sum(xs) / len(xs)


def _nearby(box_a, box_b, y_thresh_ratio=1.6):
    ya, yb = _cy(box_a), _cy(box_b)
    ha = max(p[1] for p in box_a) - min(p[1] for p in box_a)
    hb = max(p[1] for p in box_b) - min(p[1] for p in box_b)
    h = max(ha, hb, 1)
    return abs(ya - yb) < h * y_thresh_ratio


def stitch_fragments(ocr_results, x_gap_ratio=2.5, y_overlap_ratio=0.6):
    """같은 줄에서 쪼개진 조각들(예: 2022 / 02.14)을 이어붙여 추가 후보 문자열을 만든다."""
    items = list(ocr_results)
    items.sort(key=lambda t: (_cy(t[0]), _cx(t[0])))
    stitched = []
    used = [False] * len(items)
    for i in range(len(items)):
        if used[i]:
            continue
        group = [items[i]]
        used[i] = True
        cy = _cy(items[i][0])
        ch = max(p[1] for p in items[i][0]) - min(p[1] for p in items[i][0])
        last_xmax = max(p[0] for p in items[i][0])
        for j in range(i + 1, len(items)):
            if used[j]:
                continue
            bj = items[j][0]
            cyj = _cy(bj)
            xminj = min(p[0] for p in bj)
            if abs(cyj - cy) < max(ch, 1) * y_overlap_ratio and 0 <= (xminj - last_xmax) < ch * x_gap_ratio:
                group.append(items[j])
                used[j] = True
                last_xmax = max(p[0] for p in bj)
        if len(group) > 1:
            merged_text = " ".join(g[1] for g in group)
            stitched.append((group[0][0], merged_text, min(g[2] for g in group)))
    return stitched


def extract_best_date(ocr_results):
    """ocr_results: [(box[[x,y]x4], text, conf), ...] -> (year, month, day) or (None, None, None)"""
    if not ocr_results:
        return None, None, None
    country_bucket = _detect_country_bucket(ocr_results)
    all_items = list(ocr_results) + stitch_fragments(ocr_results)
    candidates = []
    for (box, text, conf) in all_items:
        dates = find_date_candidates(text, country_bucket)
        if not dates:
            continue
        context_texts = [text]
        for (box2, text2, conf2) in ocr_results:
            if box2 is box:
                continue
            if _nearby(box, box2):
                context_texts.append(text2)
        context = " ".join(context_texts)
        has_pos = any(k in context for k in POS_KEYWORDS)
        has_suffix = any(k in context for k in SUFFIX_KEYWORDS)
        has_neg = any(k in context for k in NEG_KEYWORDS)
        has_lot = any(k in text for k in LOT_KEYWORDS)
        score = 10.0
        if has_pos:
            score += 100
        if has_suffix:
            score += 60
        if has_neg:
            score -= 90
        if has_lot:
            score -= 40
        score += conf * 5
        for (y, mo, d, rank) in dates:
            candidates.append((score, rank, conf, y, mo, d))
    if not candidates:
        return None, None, None
    # 정렬 기준: (1) 키워드 점수 — 소비기한/제조일자 구분이 언제나 최우선,
    #            (2) 근거 강도 rank — 점수가 같을 때 4자리 완전 날짜 > 2자리 연도 >
    #                일부 NONE 인 부분 후보 순으로 고른다,
    #            (3) OCR 신뢰도.
    # rank 를 점수보다 앞세우지 않는 이유: 그렇게 하면 제조일자가 깨끗하게 읽히고
    # 소비기한이 일부만 읽힌 라벨에서 제조일자를 답으로 내보내게 된다.
    candidates.sort(key=lambda t: (-t[0], t[1], -t[2]))
    _, _, _, y, mo, d = candidates[0]
    return y, mo, d


## 4. OCR 파이프라인 (1단계 배치 인식 → 2단계 ROI 재시도 → 3단계 전체 재검출)

In [ ]:
import cv2
import numpy as np


def propose_line_boxes(img_bgr, work_width=FAST_WORK_WIDTH, max_width_ratio=0.45,
                        min_width_ratio=0.02, max_boxes=FAST_MAX_BOXES):
    h0, w0 = img_bgr.shape[:2]
    scale = work_width / w0
    small = cv2.resize(img_bgr, (work_width, max(1, int(h0 * scale))))
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    grad = cv2.morphologyEx(gray, cv2.MORPH_GRADIENT, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)))
    _, bw = cv2.threshold(grad, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    # 세로 방향으로는 거의 붙이지 않아(커널 높이=1) 서로 다른 줄이 하나로 합쳐지는 것을 방지
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 1))
    connected = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel, iterations=1)
    connected = cv2.dilate(connected, cv2.getStructuringElement(cv2.MORPH_RECT, (5, 2)), iterations=1)
    contours, _ = cv2.findContours(connected, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    H, W = small.shape[:2]
    cand = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if w < 18 or h < 8:
            continue
        ar = w / float(h)
        if ar < 1.3:
            continue
        if h > H * 0.15:
            continue
        wratio = w / W
        if wratio > max_width_ratio or wratio < min_width_ratio:
            continue   # 문단형 긴 줄(설명문)과 잡음 조각을 제외
        cand.append((w * h, x, y, w, h))
    cand.sort(key=lambda t: -t[0])
    cand = cand[:max_boxes]
    boxes = []
    for area, x, y, w, h in cand:
        boxes.append((x / scale, y / scale, (x + w) / scale, (y + h) / scale))
    return boxes


def _to_recognize_format(boxes):
    return [[int(x0), int(x1), int(y0), int(y1)] for (x0, y0, x1, y1) in boxes]


def run_fast_path(reader, img_bgr, gray):
    """1단계: classical CV 후보 검출 + 배치 인식 (검출기 CRAFT는 생략)."""
    boxes = propose_line_boxes(img_bgr)
    if not boxes:
        return [], []
    hlist = _to_recognize_format(boxes)
    try:
        raw = reader.recognize(gray, horizontal_list=hlist, free_list=[],
                                batch_size=min(FAST_MAX_BOXES, len(boxes)))
        return list(raw), boxes
    except Exception:
        return [], boxes


def run_roi_retry(reader, img_bgr, boxes):
    """2단계: 1단계 후보 중 상위 ROI_RETRY_BOXES개만 대비 보정해 재인식한다
    (이미지 전체를 다시 검출하지 않으므로 저비용)."""
    if not boxes:
        return []
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    enhanced = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(gray)
    hlist = _to_recognize_format(boxes[:ROI_RETRY_BOXES])
    try:
        return list(reader.recognize(enhanced, horizontal_list=hlist, free_list=[],
                                      batch_size=len(hlist), contrast_ths=0.03, adjust_contrast=0.7))
    except Exception:
        return []


def run_full_redetect(reader, img_bgr, canvas):
    """3단계: 1·2단계가 classical CV 박스 자체를 놓쳐 모두 실패했을 때만, 시간이
    허락하는 경우에 이미지 전체를 다시 검출한다 (CLAHE 대비보정 + EasyOCR 전체 파이프라인)."""
    h0, w0 = img_bgr.shape[:2]
    scale = canvas / max(h0, w0)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    small = cv2.resize(enhanced, (max(1, int(w0 * scale)), max(1, int(h0 * scale))))
    small_bgr = cv2.cvtColor(small, cv2.COLOR_GRAY2BGR)
    try:
        raw = reader.readtext(small_bgr, canvas_size=canvas)
    except Exception:
        return []
    results = []
    for (box, text, conf) in raw:
        scaled_box = [[p[0] / scale, p[1] / scale] for p in box]
        results.append((scaled_box, text, conf))
    return results


def process_one_image(reader, path, tier3_canvas):
    """tier3_canvas가 None이면 3단계(전체 재검출)를 생략한다 (시간 예산이 없을 때)."""
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        return None, None, None
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    fast_results, boxes = run_fast_path(reader, img_bgr, gray)
    y, mo, d = extract_best_date(fast_results)
    if y is not None:
        return y, mo, d
    roi_results = run_roi_retry(reader, img_bgr, boxes)
    combined = fast_results + roi_results
    y, mo, d = extract_best_date(combined)
    if y is not None or tier3_canvas is None:
        return y, mo, d
    full_results = run_full_redetect(reader, img_bgr, tier3_canvas)
    y, mo, d = extract_best_date(combined + full_results)
    return y, mo, d


## 5. 병렬 처리 + 3단계 시간 예산 가드레일

`ThreadPoolExecutor` 로 이미지 단위 작업을 워커 스레드에 분배한다. 각 스레드는 최초 호출 시
자신만의 EasyOCR Reader 를 한 번만 로드해 재사용한다(스레드-로컬). 1·2단계는 모든 이미지에
항상 시도한다(저비용). 3단계(전체 재검출)만, 공유 시작 시각과 지금까지 완료된 이미지 수
(스레드 간 락으로 보호되는 공유 카운터)로 **남은 이미지 1장당 남은 "워커-시간"**을 계산해서
— 여유가 크면 큰 canvas로, 작으면 작은 canvas로, 그마저 없으면 3단계 자체를 생략한다
(`TIER3_CANVAS_SCHEDULE`, `TIER3_MIN_VIABLE_BUDGET`). 이 여유시간 계산의 budget은 이미
`SAFETY_MARGIN_SEC`(250초)이 반영된 `TIME_BUDGET_SEC`이므로, 3단계가 아무리 적극적으로 시간을
써도 안전 마진 밑으로는 내려가지 않는다. EasyOCR/torch/OpenCV 의 실제 연산 대부분은 C/C++
구현으로 GIL 을 해제하므로, 스레드 기반으로도 프로세스 기반과 유사한 병렬 처리량을 얻으면서
`fork()` 를 아예 사용하지 않아 Jupyter 커널 환경에서의 안정성을 높인다.


In [ ]:
_thread_local = threading.local()
_progress_lock = threading.Lock()
_progress_state = {"completed": 0}


def _get_worker_reader():
    if not hasattr(_thread_local, "reader"):
        import easyocr
        try:
            import torch
            torch.set_num_threads(1)
        except Exception:
            pass
        _thread_local.reader = easyocr.Reader(
            ["ko", "en"], gpu=False,
            model_storage_directory=WEIGHTS_DIR,
            download_enabled=False,
            verbose=False,
        )
    return _thread_local.reader


def _pick_tier3_canvas(elapsed, completed, total, budget, n_workers):
    """3단계(전체 재검출)에 쓸 canvas를 고른다. n_workers 개 워커가 동시에 처리하므로,
    실제 허용 가능한 이미지당 처리 시간은 (남은 시간 * n_workers) / (남은 이미지 수) 이다
    (이 배수를 빠뜨리면 항상 최소 canvas로만 떨어지는 오류가 생긴다는 것을 확인했다).
    여유가 없으면 None을 반환해 3단계를 생략하고 1·2단계 결과(NONE일 수도 있음)를 그대로 쓴다."""
    remaining_imgs = max(1, total - completed)
    remaining_time = budget - elapsed
    if remaining_time <= 0:
        return None
    per_img_budget = remaining_time * n_workers / remaining_imgs
    for threshold, canvas in TIER3_CANVAS_SCHEDULE:
        if per_img_budget >= threshold:
            return canvas
    if per_img_budget >= TIER3_MIN_VIABLE_BUDGET:
        return TIER3_CANVAS_SCHEDULE[-1][1]
    return None


def _worker_process(path, start_time, budget, total_images, n_workers):
    image_id = os.path.splitext(os.path.basename(path))[0]
    elapsed = time.time() - start_time
    if elapsed > budget:
        # 안전 마진 반영 예산 초과: 1·2단계조차 생략하고 안전하게 NONE 처리
        return {"image_id": image_id, "year": "NONE", "month": "NONE", "day": "NONE", "final_date": "NONE-NONE-NONE"}
    with _progress_lock:
        completed = _progress_state["completed"]
    tier3_canvas = _pick_tier3_canvas(elapsed, completed, total_images, budget, n_workers)
    try:
        reader = _get_worker_reader()
        y, mo, d = process_one_image(reader, path, tier3_canvas)
    except Exception:
        y, mo, d = None, None, None
    with _progress_lock:
        _progress_state["completed"] += 1
    if y is None:
        return {"image_id": image_id, "year": "NONE", "month": "NONE", "day": "NONE", "final_date": "NONE-NONE-NONE"}
    return {"image_id": image_id, "year": y, "month": mo, "day": d, "final_date": f"{y}-{mo}-{d}"}


## 6. 실행: INPUT_DIR 의 모든 이미지에 대해 추론

In [ ]:
image_files = sorted(
    glob.glob(os.path.join(INPUT_DIR, "*.jpg")) +
    glob.glob(os.path.join(INPUT_DIR, "*.jpeg")) +
    glob.glob(os.path.join(INPUT_DIR, "*.png")) +
    glob.glob(os.path.join(INPUT_DIR, "*.bmp")) +
    glob.glob(os.path.join(INPUT_DIR, "*.webp")) +
    glob.glob(os.path.join(INPUT_DIR, "*.JPG")) +
    glob.glob(os.path.join(INPUT_DIR, "*.JPEG")) +
    glob.glob(os.path.join(INPUT_DIR, "*.PNG")) +
    glob.glob(os.path.join(INPUT_DIR, "*.BMP")) +
    glob.glob(os.path.join(INPUT_DIR, "*.WEBP"))
)
print(f"input images: {len(image_files)}")

results = []
if len(image_files) == 0:
    print("경고: INPUT_DIR 에서 이미지를 찾지 못했습니다.")
else:
    start_time = time.time()
    total_images = len(image_files)
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = [executor.submit(_worker_process, p, start_time, TIME_BUDGET_SEC, total_images, N_WORKERS) for p in image_files]
        for i, fut in enumerate(futures):
            results.append(fut.result())
            if (i + 1) % 200 == 0:
                print(f"  processed {i+1}/{len(image_files)}  elapsed={time.time()-start_time:.1f}s")
    print(f"done. total elapsed={time.time()-start_time:.1f}s")


## 7. submission.csv 저장

In [ ]:
import pandas as pd

df = pd.DataFrame(results, columns=["image_id", "year", "month", "day", "final_date"])
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")
print(df.head())
print("NONE ratio:", (df["final_date"] == "NONE-NONE-NONE").mean())
